# CFLOW Two-Stage (Stable, Colab)

This notebook is aligned to the actual `gudovskiy/cflow-ad` code behavior:
- uses MVTec class `bottle` only (repo constraint)
- runs CFLOW through `norm-train` path
- exports real per-image stage-1 score vectors
- runs tiny pilot end-to-end before full run


In [ ]:
# 0) One-time ABI fix (run then restart runtime)
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'numpy==1.26.4'])
print('Installed numpy==1.26.4. Restart runtime now, then run from cell 1.')


In [ ]:
# 1) Runtime preflight
import sys, numpy as np, torch, torchvision
print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('Torch:', torch.__version__)
print('TorchVision:', torchvision.__version__)
assert np.__version__.startswith('1.26'), 'Need numpy==1.26.x'
print('Preflight OK')


In [ ]:
# 2) Setup repos + deps + minimal compatibility
import os, sys, subprocess
from pathlib import Path

FYP_REPO = Path('/content/FYP-code')
if not FYP_REPO.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/spinelessknave8/FYP_code.git', str(FYP_REPO)])
subprocess.check_call(['git', '-C', str(FYP_REPO), 'fetch', 'origin'])
subprocess.check_call(['git', '-C', str(FYP_REPO), 'reset', '--hard', 'origin/main'])

CFLOW_REPO = Path('/content/cflow-ad')
if not CFLOW_REPO.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/gudovskiy/cflow-ad.git', str(CFLOW_REPO)])
subprocess.check_call(['git', '-C', str(CFLOW_REPO), 'fetch', 'origin'])
subprocess.check_call(['git', '-C', str(CFLOW_REPO), 'reset', '--hard', 'origin/master'])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm==0.9.7', 'FrEIA==0.2', 'scikit-learn==1.3.2'])

# Pillow compatibility
loader_py = CFLOW_REPO / 'custom_datasets' / 'loader.py'
loader_txt = loader_py.read_text(errors='ignore')
if 'Image.ANTIALIAS' in loader_txt:
    loader_py.write_text(loader_txt.replace('Image.ANTIALIAS', 'Image.Resampling.LANCZOS'))

subprocess.check_call([sys.executable, '-m', 'py_compile', str(loader_py)])
subprocess.check_call([sys.executable, '-m', 'py_compile', str(CFLOW_REPO / 'main.py')])

os.chdir(FYP_REPO)
for p in [str(FYP_REPO), str(FYP_REPO/'src'), str(FYP_REPO/'severstral-osr'/'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('Setup complete')


In [ ]:
# 3) Drive + dataset checks
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

DATA_ROOT = Path('/content/drive/MyDrive/datasets/severstal')
TRAIN_CSV = DATA_ROOT / 'train.csv'
TRAIN_IMAGES = DATA_ROOT / 'train_images'
OUT_ROOT = Path('/content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full')
OUT_ROOT.mkdir(parents=True, exist_ok=True)

assert TRAIN_CSV.exists(), f'Missing: {TRAIN_CSV}'
assert TRAIN_IMAGES.exists() and TRAIN_IMAGES.is_dir(), f'Missing: {TRAIN_IMAGES}'
print('OUT_ROOT:', OUT_ROOT)


In [ ]:
# 4) Build/reuse full split
import csv, json, random
from collections import defaultdict

SPLIT_PATH = OUT_ROOT / 'split_manifest.json'
FORCE_REBUILD_SPLIT = False

if SPLIT_PATH.exists() and not FORCE_REBUILD_SPLIT:
    split = json.loads(SPLIT_PATH.read_text())
else:
    SEED=42
    KNOWN=['Class_1','Class_2','Class_3']
    rng=random.Random(SEED)

    rows_by_img=defaultdict(list)
    with open(TRAIN_CSV,'r',newline='') as f:
        rd=csv.DictReader(f)
        for r in rd:
            rows_by_img[r['ImageId'].strip()].append(r)

    imgs=[p for p in TRAIN_IMAGES.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'}]
    normals=[]; single=[]
    for p in imgs:
        pos=set()
        for r in rows_by_img.get(p.name,[]):
            enc=str(r.get('EncodedPixels','')).strip().lower()
            if enc and enc!='nan': pos.add(int(r['ClassId']))
        if len(pos)==0: normals.append(str(p))
        elif len(pos)==1: single.append((str(p), f"Class_{next(iter(pos))}"))

    known_all=[(p,c) for p,c in single if c in KNOWN]
    unk_all=[(p,c) for p,c in single if c not in KNOWN]

    def split_cls(samples, seed=42):
        rr=random.Random(seed)
        by=defaultdict(list)
        for s in samples: by[s[1]].append(s)
        tr,va,te=[],[],[]
        for items in by.values():
            rr.shuffle(items)
            n=len(items); ntr=int(n*0.7); nva=int(n*0.15)
            tr+=items[:ntr]; va+=items[ntr:ntr+nva]; te+=items[ntr+nva:]
        return tr,va,te

    ktr,kva,kte=split_cls(known_all, seed=SEED)
    rng.shuffle(normals); rng.shuffle(unk_all)

    split={
      'seed':SEED,
      'known_classes':KNOWN,
      'normal_train':normals[:1200],
      'normal_val':normals[1200:1500],
      'normal_test':normals[1500:1800],
      'known_train':ktr[:900],
      'known_val':kva[:300],
      'known_test':kte[:300],
      'unknown_test':unk_all[:300],
    }
    SPLIT_PATH.write_text(json.dumps(split, indent=2))

print({k:len(v) for k,v in split.items() if isinstance(v,list)})


In [ ]:
# 5) Build filesystem datasets for FULL + PILOT (outside MVTec-AD), then we swap into bottle runtime
import json, shutil
from pathlib import Path
from PIL import Image
import numpy as np

split = json.loads((OUT_ROOT/'split_manifest.json').read_text())

BUILD = Path('/content/cflow-ad/data/_severstal_build')
if BUILD.exists():
    shutil.rmtree(BUILD)
BUILD.mkdir(parents=True, exist_ok=True)

P = {
  'normal_train': 96,
  'normal_val': 32,
  'normal_test': 32,
  'known_train': 96,
  'known_val': 32,
  'known_test': 32,
  'unknown_test': 32,
  'val_defect_count': 8,
}
pilot = {
  'normal_train': split['normal_train'][:P['normal_train']],
  'normal_val': split['normal_val'][:P['normal_val']],
  'normal_test': split['normal_test'][:P['normal_test']],
  'known_train': split['known_train'][:P['known_train']],
  'known_val': split['known_val'][:P['known_val']],
  'known_test': split['known_test'][:P['known_test']],
  'unknown_test': split['unknown_test'][:P['unknown_test']],
}


def build_pair(tag, src_split, val_defect_count):
    main = BUILD / f'{tag}_main'
    val = BUILD / f'{tag}_val'
    for r in [main, val]:
        if r.exists(): shutil.rmtree(r)
    for p in [
      main/'train'/'good', main/'test'/'good', main/'test'/'known_defect', main/'test'/'unknown_defect',
      main/'ground_truth'/'known_defect', main/'ground_truth'/'unknown_defect',
      val/'train'/'good', val/'test'/'good', val/'test'/'known_defect', val/'ground_truth'/'known_defect',
    ]:
      p.mkdir(parents=True, exist_ok=True)

    def write_png(rows, dst, labeled=False):
      n=0
      for i,row in enumerate(rows):
        src=Path(row[0] if labeled else row)
        if not src.exists():
          continue
        out=dst/f'{i:06d}.png'
        try:
          Image.open(src).convert('RGB').save(out, format='PNG'); n+=1
        except Exception:
          pass
      return n

    def mk_masks(tdir, gdir):
      for img in sorted(tdir.glob('*.png')):
        m=gdir/f'{img.stem}_mask.png'
        if m.exists():
          continue
        w,h=Image.open(img).size
        Image.fromarray((np.ones((h,w),dtype=np.uint8)*255)).save(m, format='PNG')

    c={}
    c['main_train_good']=write_png(src_split['normal_train'], main/'train'/'good')
    c['main_test_good']=write_png(src_split['normal_test'], main/'test'/'good')
    c['main_test_known_defect']=write_png(src_split['known_test'], main/'test'/'known_defect', labeled=True)
    c['main_test_unknown_defect']=write_png(src_split['unknown_test'], main/'test'/'unknown_defect', labeled=True)
    mk_masks(main/'test'/'known_defect', main/'ground_truth'/'known_defect')
    mk_masks(main/'test'/'unknown_defect', main/'ground_truth'/'unknown_defect')

    c['val_train_good']=write_png(src_split['normal_train'], val/'train'/'good')
    c['val_test_good']=write_png(src_split['normal_val'], val/'test'/'good')
    c['val_test_known_defect']=write_png(src_split['known_test'][:val_defect_count], val/'test'/'known_defect', labeled=True)
    mk_masks(val/'test'/'known_defect', val/'ground_truth'/'known_defect')

    assert min(c.values()) > 0, f'Empty dataset partitions for {tag}: {c}'
    return {'main': str(main), 'val': str(val), 'counts': c, 'val_defect_count': int(c['val_test_known_defect'])}

full_meta = build_pair('full', split, val_defect_count=20)
pilot_meta = build_pair('pilot', pilot, val_defect_count=P['val_defect_count'])

(OUT_ROOT/'dataset_meta.json').write_text(json.dumps({'full': full_meta, 'pilot': pilot_meta, 'pilot_sizes': P}, indent=2))
print(json.dumps({'full': full_meta['counts'], 'pilot': pilot_meta['counts']}, indent=2))


In [ ]:
# 6) Patch train.py safely (reset + one-line export patch + compile)
import subprocess, sys
from pathlib import Path

cwd='/content/cflow-ad'
subprocess.check_call(['git','checkout','--','train.py'], cwd=cwd)

train_py=Path('/content/cflow-ad/train.py')
txt=train_py.read_text(errors='ignore')
anchor='det_roc_auc = roc_auc_score(gt_label, score_label)'
marker='scores_run{c.run_name}_{c.class_name}_epoch{epoch}.npz'

if anchor not in txt:
    raise RuntimeError('Anchor not found in train.py')
if marker not in txt:
    patched = anchor + "; os.makedirs('./results', exist_ok=True); np.savez(os.path.join('./results', f'scores_run{c.run_name}_{c.class_name}_epoch{epoch}.npz'), score_label=np.asarray(score_label), gt_label=np.asarray(gt_label))"
    txt = txt.replace(anchor, patched, 1)
    train_py.write_text(txt)

subprocess.check_call([sys.executable, '-m', 'py_compile', str(train_py)])
print('train.py patched + compiled')


In [ ]:
# 7) Tiny PILOT end-to-end (real metrics schema)
import subprocess, os, shutil, json
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from src.models.resnet50 import build_resnet50
from sklearn.metrics import roc_auc_score

meta=json.loads((OUT_ROOT/'dataset_meta.json').read_text())
ps=meta['pilot_sizes']
pilot_meta=meta['pilot']
split=json.loads((OUT_ROOT/'split_manifest.json').read_text())

pilot={
 'normal_train': split['normal_train'][:ps['normal_train']],
 'normal_val': split['normal_val'][:ps['normal_val']],
 'normal_test': split['normal_test'][:ps['normal_test']],
 'known_train': split['known_train'][:ps['known_train']],
 'known_val': split['known_val'][:ps['known_val']],
 'known_test': split['known_test'][:ps['known_test']],
 'unknown_test': split['unknown_test'][:ps['unknown_test']],
 'known_classes': split['known_classes'],
}

BASE=Path('/content/cflow-ad/data/MVTec-AD')
runtime_main=BASE/'bottle'
runtime_backup=BASE/'bottle_backup_runtime'
if runtime_backup.exists(): shutil.rmtree(runtime_backup)

# swap pilot main into runtime bottle
if runtime_main.exists(): os.rename(runtime_main, runtime_backup)
shutil.copytree(pilot_meta['main'], runtime_main)

cwd='/content/cflow-ad'
RUN_MAIN=3001
subprocess.check_call([
  'python','main.py','--dataset','mvtec','--class-name','bottle','--action-type','norm-train',
  '--run-name',str(RUN_MAIN),'--batch-size','8','--meta-epochs','1','--sub-epochs','1','--workers','2','--input-size','256','--gpu','0'
], cwd=cwd)

# swap pilot val into runtime bottle and run val calibration pass
if runtime_main.exists(): shutil.rmtree(runtime_main)
shutil.copytree(pilot_meta['val'], runtime_main)
RUN_VAL=3002
subprocess.check_call([
  'python','main.py','--dataset','mvtec','--class-name','bottle','--action-type','norm-train',
  '--run-name',str(RUN_VAL),'--batch-size','8','--meta-epochs','1','--sub-epochs','1','--workers','2','--input-size','256','--gpu','0'
], cwd=cwd)

# restore runtime bottle
if runtime_main.exists(): shutil.rmtree(runtime_main)
os.rename(runtime_backup, runtime_main)

res=Path('/content/cflow-ad/results')
main_npz=sorted(res.glob(f'scores_run{RUN_MAIN}_bottle_epoch*.npz'))
val_npz=sorted(res.glob(f'scores_run{RUN_VAL}_bottle_epoch*.npz'))
if not main_npz or not val_npz:
  raise RuntimeError('Pilot score artifacts missing')

main_scores=np.load(main_npz[-1])['score_label'].reshape(-1)
val_scores=np.load(val_npz[-1])['score_label'].reshape(-1)

n_g=len(pilot['normal_test']); n_k=len(pilot['known_test']); n_u=len(pilot['unknown_test']); n_vg=len(pilot['normal_val'])
s_nts=main_scores[:n_g]; s_kts=main_scores[n_g:n_g+n_k]; s_uts=main_scores[n_g+n_k:n_g+n_k+n_u]; s_nva=val_scores[:n_vg]

# stage2 pilot
class_to_idx={c:i for i,c in enumerate(pilot['known_classes'])}
device='cuda' if torch.cuda.is_available() else 'cpu'
tf=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

class KnownDS(Dataset):
  def __init__(self, rows): self.rows=rows
  def __len__(self): return len(self.rows)
  def __getitem__(self,i):
    p,c=self.rows[i]
    return tf(Image.open(p).convert('RGB')), class_to_idx[c]

tr=DataLoader(KnownDS(pilot['known_train']), batch_size=16, shuffle=True, num_workers=2)
va=DataLoader(KnownDS(pilot['known_val']), batch_size=16, shuffle=False, num_workers=2)
model=build_resnet50(num_classes=len(class_to_idx), pretrained=True).to(device)
opt=torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
crit=nn.CrossEntropyLoss(); best_va=-1.0; best=None
for _ in range(1):
  model.train()
  for x,y in tr:
    x,y=x.to(device),y.to(device)
    opt.zero_grad(); loss=crit(model(x),y); loss.backward(); opt.step()
  model.eval(); cor=tot=0
  with torch.no_grad():
    for x,y in va:
      p=model(x.to(device)).argmax(1).cpu(); cor += (p==y).sum().item(); tot += len(y)
  acc=cor/max(1,tot)
  if acc>best_va: best_va=acc; best={k:v.cpu() for k,v in model.state_dict().items()}
model.load_state_dict(best)

def logits(rows):
  class DS(Dataset):
    def __init__(self, rows): self.rows=rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
      r=self.rows[i]; p=r[0] if isinstance(r,(tuple,list)) else r
      return tf(Image.open(p).convert('RGB'))
  dl=DataLoader(DS(rows), batch_size=16, shuffle=False, num_workers=2)
  out=[]; model.eval()
  with torch.no_grad():
    for x in dl: out.append(model(x.to(device)).cpu().numpy())
  return np.concatenate(out,0)

lv=logits(pilot['known_val']); lk=logits(pilot['known_test']); lu=logits(pilot['unknown_test'])

def margin(logits):
  p=np.partition(logits, -2, axis=1)
  return p[:,-1]-p[:,-2]

m_val=margin(lv); m_k=margin(lk); m_u=margin(lu)
pk=lk.argmax(1); yk=np.array([class_to_idx[c] for _,c in pilot['known_test']])
auroc=float(roc_auc_score(np.r_[np.zeros(len(s_nts)), np.ones(len(s_kts)+len(s_uts))], np.r_[s_nts,s_kts,s_uts]))

rows=[]
for fpr in [0.05,0.10,0.20]:
  tau=float(np.quantile(s_nva, 1-fpr)); kappa=float(np.quantile(m_val, fpr))
  n_def=s_nts>tau; k_def=s_kts>tau; u_def=s_uts>tau
  k_unk=m_k<kappa; u_unk=m_u<kappa
  k_succ=k_def & (~k_unk) & (pk==yk); u_succ=u_def & u_unk
  rows.append({'split':'pilot','fpr_target':fpr,'AUROC_defect_screening':auroc,'TPR_defect@FPR':float(np.mean(np.r_[k_def,u_def])),'TPR_unknown@FPR':float(np.mean(u_def)),'FPR_known_as_def@FPR':float(np.mean(k_def)),'FPR_normal_realized':float(np.mean(n_def)),'two_stage_known_success':float(np.mean(k_succ)),'two_stage_unknown_success':float(np.mean(u_succ))})

pilot_df=pd.DataFrame(rows)
pilot_df.to_csv(OUT_ROOT/'pilot_two_stage_summary.csv', index=False)
print('Pilot summary saved:', OUT_ROOT/'pilot_two_stage_summary.csv')
display(pilot_df)


In [ ]:
# 8) Full stage-1 runs (swap full datasets into runtime bottle)
import subprocess, os, shutil, json
from pathlib import Path

meta=json.loads((OUT_ROOT/'dataset_meta.json').read_text())
full=meta['full']
BASE=Path('/content/cflow-ad/data/MVTec-AD')
runtime_main=BASE/'bottle'
runtime_backup=BASE/'bottle_backup_runtime'
if runtime_backup.exists(): shutil.rmtree(runtime_backup)

if runtime_main.exists(): os.rename(runtime_main, runtime_backup)
shutil.copytree(full['main'], runtime_main)

cwd='/content/cflow-ad'
RUN_MAIN=4001
subprocess.check_call([
  'python','main.py','--dataset','mvtec','--class-name','bottle','--action-type','norm-train',
  '--run-name',str(RUN_MAIN),'--batch-size','16','--meta-epochs','8','--sub-epochs','4','--workers','2','--input-size','256','--gpu','0'
], cwd=cwd)

if runtime_main.exists(): shutil.rmtree(runtime_main)
shutil.copytree(full['val'], runtime_main)
RUN_VAL=4002
subprocess.check_call([
  'python','main.py','--dataset','mvtec','--class-name','bottle','--action-type','norm-train',
  '--run-name',str(RUN_VAL),'--batch-size','16','--meta-epochs','1','--sub-epochs','1','--workers','2','--input-size','256','--gpu','0'
], cwd=cwd)

if runtime_main.exists(): shutil.rmtree(runtime_main)
os.rename(runtime_backup, runtime_main)

(OUT_ROOT/'full_run_ids.json').write_text(json.dumps({'main':RUN_MAIN,'val':RUN_VAL}, indent=2))
print('Full stage1 runs complete')


In [ ]:
# 9) Load full stage-1 real score vectors
import json, numpy as np
from pathlib import Path

split=json.loads((OUT_ROOT/'split_manifest.json').read_text())
ids=json.loads((OUT_ROOT/'full_run_ids.json').read_text())
res=Path('/content/cflow-ad/results')

main_npz=sorted(res.glob(f"scores_run{ids['main']}_bottle_epoch*.npz"))
val_npz=sorted(res.glob(f"scores_run{ids['val']}_bottle_epoch*.npz"))
if not main_npz: raise RuntimeError('Missing full main stage1 score npz')
if not val_npz: raise RuntimeError('Missing full val stage1 score npz')

main_scores=np.load(main_npz[-1])['score_label'].reshape(-1)
val_scores=np.load(val_npz[-1])['score_label'].reshape(-1)

n_g=len(split['normal_test']); n_k=len(split['known_test']); n_u=len(split['unknown_test']); n_vg=len(split['normal_val'])
if len(main_scores)!=(n_g+n_k+n_u):
  raise RuntimeError(f'Main score length mismatch: got {len(main_scores)} expected {n_g+n_k+n_u}')
if len(val_scores)<n_vg:
  raise RuntimeError(f'Val score length too small: got {len(val_scores)} expected at least {n_vg}')

s_nts=main_scores[:n_g]
s_kts=main_scores[n_g:n_g+n_k]
s_uts=main_scores[n_g+n_k:n_g+n_k+n_u]
s_nva=val_scores[:n_vg]

np.save(OUT_ROOT/'stage1_scores_normal_val.npy', s_nva)
np.save(OUT_ROOT/'stage1_scores_normal_test.npy', s_nts)
np.save(OUT_ROOT/'stage1_scores_known_test.npy', s_kts)
np.save(OUT_ROOT/'stage1_scores_unknown_test.npy', s_uts)
print('Saved full stage1 vectors')


In [ ]:
# 10) Stage-2 full ResNet50 classifier
import json, numpy as np, torch, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from src.models.resnet50 import build_resnet50

split=json.loads((OUT_ROOT/'split_manifest.json').read_text())
class_to_idx={c:i for i,c in enumerate(split['known_classes'])}
device='cuda' if torch.cuda.is_available() else 'cpu'

tf=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

class KnownDS(Dataset):
  def __init__(self, rows): self.rows=rows
  def __len__(self): return len(self.rows)
  def __getitem__(self,i):
    p,c=self.rows[i]
    return tf(Image.open(p).convert('RGB')), class_to_idx[c]

tr=DataLoader(KnownDS(split['known_train']), batch_size=32, shuffle=True, num_workers=2)
va=DataLoader(KnownDS(split['known_val']), batch_size=32, shuffle=False, num_workers=2)

model=build_resnet50(num_classes=len(class_to_idx), pretrained=True).to(device)
opt=torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
crit=nn.CrossEntropyLoss(); best_va=-1.0; best=None

for ep in range(4):
  model.train()
  for x,y in tr:
    x,y=x.to(device),y.to(device)
    opt.zero_grad(); loss=crit(model(x),y); loss.backward(); opt.step()
  model.eval(); cor=tot=0
  with torch.no_grad():
    for x,y in va:
      p=model(x.to(device)).argmax(1).cpu(); cor += (p==y).sum().item(); tot += len(y)
  acc=cor/max(1,tot)
  print(f'epoch {ep+1}/4 val_acc={acc:.4f}')
  if acc>best_va: best_va=acc; best={k:v.cpu() for k,v in model.state_dict().items()}

model.load_state_dict(best)

def logits(rows):
  class DS(Dataset):
    def __init__(self, rows): self.rows=rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
      r=self.rows[i]; p=r[0] if isinstance(r,(tuple,list)) else r
      return tf(Image.open(p).convert('RGB'))
  dl=DataLoader(DS(rows), batch_size=32, shuffle=False, num_workers=2)
  out=[]; model.eval()
  with torch.no_grad():
    for x in dl: out.append(model(x.to(device)).cpu().numpy())
  return np.concatenate(out,0)

np.save(OUT_ROOT/'stage2_logits_known_val.npy', logits(split['known_val']))
np.save(OUT_ROOT/'stage2_logits_known_test.npy', logits(split['known_test']))
np.save(OUT_ROOT/'stage2_logits_unknown_test.npy', logits(split['unknown_test']))
print('Saved stage2 logits')


In [ ]:
# 11) Final full two-stage metrics (write-up table)
import json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

split=json.loads((OUT_ROOT/'split_manifest.json').read_text())
class_to_idx={c:i for i,c in enumerate(split['known_classes'])}

def margin(logits):
  p=np.partition(logits, -2, axis=1)
  return p[:,-1]-p[:,-2]

s_nva=np.load(OUT_ROOT/'stage1_scores_normal_val.npy')
s_nts=np.load(OUT_ROOT/'stage1_scores_normal_test.npy')
s_kts=np.load(OUT_ROOT/'stage1_scores_known_test.npy')
s_uts=np.load(OUT_ROOT/'stage1_scores_unknown_test.npy')

lv=np.load(OUT_ROOT/'stage2_logits_known_val.npy')
lk=np.load(OUT_ROOT/'stage2_logits_known_test.npy')
lu=np.load(OUT_ROOT/'stage2_logits_unknown_test.npy')

m_val=margin(lv); m_k=margin(lk); m_u=margin(lu)
pk=lk.argmax(1); yk=np.array([class_to_idx[c] for _,c in split['known_test']])

auroc=float(roc_auc_score(np.r_[np.zeros(len(s_nts)), np.ones(len(s_kts)+len(s_uts))], np.r_[s_nts,s_kts,s_uts]))
rows=[]
for fpr in [0.05,0.10,0.20]:
  tau=float(np.quantile(s_nva, 1-fpr)); kappa=float(np.quantile(m_val, fpr))
  n_def=s_nts>tau; k_def=s_kts>tau; u_def=s_uts>tau
  k_unk=m_k<kappa; u_unk=m_u<kappa
  k_succ=k_def & (~k_unk) & (pk==yk); u_succ=u_def & u_unk
  rows.append({
    'fpr_target':fpr,'AUROC_defect_screening':auroc,
    'TPR_defect@FPR':float(np.mean(np.r_[k_def,u_def])),'TPR_unknown@FPR':float(np.mean(u_def)),
    'FPR_known_as_def@FPR':float(np.mean(k_def)),'FPR_normal_realized':float(np.mean(n_def)),
    'two_stage_known_success':float(np.mean(k_succ)),'two_stage_unknown_success':float(np.mean(u_succ)),
    'stage2_unknown_rate_known_all':float(np.mean(k_unk)),'stage2_unknown_rate_unknown_all':float(np.mean(u_unk)),
  })

res=pd.DataFrame(rows)
res.to_csv(OUT_ROOT/'cflow_two_stage_summary.csv', index=False)
print('Saved:', OUT_ROOT/'cflow_two_stage_summary.csv')
display(res)


In [ ]:
# 12) Plot summary
import pandas as pd, matplotlib.pyplot as plt
res=pd.read_csv(OUT_ROOT/'cflow_two_stage_summary.csv')
plt.figure(figsize=(7,4))
plt.plot(res['fpr_target'], res['TPR_defect@FPR'], marker='o', label='TPR_defect')
plt.plot(res['fpr_target'], res['TPR_unknown@FPR'], marker='o', label='TPR_unknown')
plt.plot(res['fpr_target'], res['FPR_normal_realized'], marker='o', label='FPR_normal')
plt.ylim(0,1); plt.xlabel('FPR target'); plt.ylabel('rate'); plt.legend()
plt.tight_layout()
plt.savefig(OUT_ROOT/'plot_cflow_two_stage_rates.png', dpi=140)
plt.show()
print('Saved:', OUT_ROOT/'plot_cflow_two_stage_rates.png')


## Run Order

After runtime restart:
1. Run cells `1 -> 12` in order.
2. Cell `7` is the tiny pilot and must pass before full run.
3. Final write-up table is cell `11` and `cflow_two_stage_summary.csv`.
